In [9]:
import sys
sys.path.append('/mnt/c/Users/JYB/ws/DeePC-Hunt')  

[autoreload of torch._custom_class_base failed: Traceback (most recent call last):
  File "/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 365, in update_class
    update_instances(old, new)
  File "/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 323, in update_instances
    object.__setattr__(ref, "__class__", new)
TypeError: can't apply this __setattr__ to CustomClassBaseMeta object
]
[autoreload of torch.overrides failed: Traceback (most r

: 

In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import torch
from numpy import genfromtxt
import torch
from deepc_hunt.dynamics import AffineDynamics
from deepc_hunt import DeePC, Trainer

# Temperature Control System

### Load in data

In [ ]:
ud = genfromtxt('data/recht_ud.csv', delimiter=',')
yd = genfromtxt('data/recht_yd.csv', delimiter=',')

# Add noise to simulate uncertainty in data
noise_std = 0.1              
yd += np.random.rand(*yd.shape)*noise_std
ud += np.random.rand(*ud.shape)*noise_std

### Initialitse DeePC controller

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

n = 3 # n = number of states
m = 3 # m = number of inputs
p = 3 # p = number of output
q = m+p # q = number of i/o variables
Tini = 4 # Past time horizon                                           
Tf = 10 # Future time horizon         
T = (m+1)*(Tini + Tf + n) - 1    

y_constraints = np.kron(np.ones(Tf), np.array([10,10,10]))
u_constraints = np.kron(np.ones(Tf), np.array([5,5,5]))
q = torch.ones(3)*50
r = torch.ones(3)*2
n_batch = 16

controller = DeePC(
    ud=ud, yd=yd, N=Tf, Tini=Tini, p=3, m=3, n_batch=n_batch, device=device,
    y_constraints=y_constraints, u_constraints=u_constraints,
    stochastic_y=True, stochastic_u=True, linear=True, q=q, r=r
)

controller.initialise(lam_y=1, lam_u=1)
controller.to(device)

/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:484: UserWarning: Found GPU0 NVIDIA TITAN Xp which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.14.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/mnt/c/Users/JYB/ws/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:602: UserWarning: 
NVIDIA TITAN Xp with CUDA capability sm_

DeePC(
  (QP_layer): CvxpyLayer()
)

### Get dynamics

In [ ]:
A = torch.Tensor([[1.01, 0.01, 0.00], # A - State-space matrix
                  [0.01, 1.01, 0.01], 
                  [0.00, 0.01, 1.01]])
dx = AffineDynamics(A=A, B=torch.eye(3)).to(device)

### Run DeePC-HUNT

In [ ]:
epochs = 2
time_steps = 10

# Tune regularization params
deepc_tuner = Trainer(controller=controller, env=dx)
final_params = deepc_tuner.run(epochs=epochs, time_steps=time_steps)

  0%|                                                                         | 0/2 [00:00<?, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
